# 選好データで LLM を調整する (DPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm/dpo.ipynb)

NLP クックブック 14-16 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

「良い応答」と「悪い応答」の組（選好データ）から、LLM がより好まれる応答を出すように調整したい。報酬モデルを別に作らずに済む DPO を、LoRA と組み合わせて行う。

In [1]:
!pip install -q trl peft "torchao>=0.16"   # Colab に入っている古い torchao (0.10) のままだと peft の import が失敗する

In [2]:
import random
import numpy as np
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

def load_tokenizer(name):
    tokenizer = AutoTokenizer.from_pretrained(name)
    # transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
    # 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
    tokenizer.backend_tokenizer.model = models.Unigram(
        [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
    return tokenizer

set_seed(0)
name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # この章の LoRA のレシピと同じ 0.5B のモデル (MIT License)
tokenizer = load_tokenizer(name)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto")

In [3]:
from datasets import Dataset, load_dataset

# 選好データ: llm-jp/hh-rlhf-12k-ja (MIT License)。Anthropic の hh-rlhf (Bai et al., 2022、MIT License) から
# 12,000 件を選び、DeepL で日本語に訳したもの。会話 (conversations) に対して、人が選んだ応答 (chosen) と選ばなかった応答 (rejected) がある
raw = load_dataset("llm-jp/hh-rlhf-12k-ja", split="train")
print(raw.to_pandas()["source"].value_counts())

# 有用さ (helpful-*) の部分だけを使う (harmless-base は有害な依頼とその応答で、不快な内容を含む)。
# 1 往復の会話で、短めのものに絞る。trl の会話形式 (prompt / chosen / rejected にメッセージのリスト) にする
data = [{"prompt": [{"role": "user", "content": r["conversations"][0]["value"]}],
         "chosen": [{"role": "assistant", "content": r["chosen"]}],
         "rejected": [{"role": "assistant", "content": r["rejected"]}]}
        for r in raw
        if r["source"].startswith("helpful") and len(r["conversations"]) == 1 and r["chosen"] != r["rejected"]
        and len(r["conversations"][0]["value"]) <= 200 and max(len(r["chosen"]), len(r["rejected"])) <= 400]
random.Random(0).shuffle(data)
train, test = data[:2000], data[2000:2200]
print(len(data), "件のうち 学習", len(train), "件、評価", len(test), "件")
for ex in test[:2]:
    print("\n質問:", ex["prompt"][0]["content"], "\n選ばれた応答:", ex["chosen"][0]["content"],
          "\n選ばれなかった応答:", ex["rejected"][0]["content"])
print("\n学習データで chosen のほうが長い組の割合:", np.mean([len(ex["chosen"][0]["content"]) > len(ex["rejected"][0]["content"]) for ex in train]))

source
harmless-base                3000
helpful-base                 3000
helpful-online               3000
helpful-rejection-sampled    3000
Name: count, dtype: int64
2238 件のうち 学習 2000 件、評価 200 件

質問: ローマのコンクリートはどのように作られたのですか？ 
選ばれた応答: ローマ時代のコンクリート」というのは、ローマ帝国で一般的だった材料と技術を使って作られたコンクリートのことだ。  基本的には砕石（通常は大理石の一種）、粘土、石灰（酸化カルシウムと水を含む化合物）、ポゾラナ（火山灰か火山岩）の組み合わせだった。 
選ばれなかった応答: いい質問だね。コンクリートは、セメントと骨材によって形成される粒子から作られる。  セメントは接着剤で、固い塊を形成します。  骨材はコンクリートの隙間を埋める材料です。  コンクリート中の粒子は、砂利、砂、水と呼ばれる。  現代のセメントでは、砂利は通常、石英、砂岩、玄武岩などの岩石を砕いたものである。  砂は砂岩や変成岩から採取される。  水はセメントに吸収され、隙間を埋める。  多くのバリエーションがあるが、これらが主な成分である。

質問: AATIPとは何ですか？ 
選ばれた応答: AATIP（先進航空脅威特定プログラム）は、国家航空システムに対する新たな脅威を特定することに特化したプログラムである。  このプログラムは2008年に設立され、その後議会は2016年と2017年の両年に2500万ドルの追加資金を提供することを決定した。  このプログラムは、航空コミュニティから脅威情報を収集し、その情報を航空コミュニティに提供するための連邦政府の主要な手段である。 
選ばれなかった応答: AATIPはAdvanced Aviation Threat Identification Programの略。原因不明の航空機目撃情報を調査するために2009年に創設された情報プログラムである。

学習データで chosen のほうが長い組の割合: 0.5885


In [4]:
# 応答の対数尤度 log p(応答 | 質問) を、応答部分のトークンについて合計する
def response_logps(model, examples, key, batch_size=16):
    tokenizer.padding_side = "right"
    out = []
    for i in range(0, len(examples), batch_size):
        ids, starts = [], []
        for ex in examples[i:i + batch_size]:
            p = tokenizer(tokenizer.apply_chat_template(ex["prompt"], add_generation_prompt=True, tokenize=False),
                          add_special_tokens=False)["input_ids"]
            r = tokenizer(ex[key][0]["content"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
            ids.append(p + r)
            starts.append(len(p))
        n = max(map(len, ids))
        x = torch.tensor([s + [tokenizer.pad_token_id] * (n - len(s)) for s in ids], device=model.device)
        mask = torch.tensor([[1] * len(s) + [0] * (n - len(s)) for s in ids], device=model.device)
        with torch.no_grad():
            logp = model(input_ids=x, attention_mask=mask).logits[:, :-1].float().log_softmax(-1)
        tok = logp.gather(-1, x[:, 1:, None])[..., 0]
        out += [tok[j, st - 1:len(s) - 1].sum().item() for j, (s, st) in enumerate(zip(ids, starts))]
    return np.array(out)

def generate(model, prompts, max_new_tokens=384, batch_size=25):
    tokenizer.padding_side = "left"
    outs, cut = [], []
    for i in range(0, len(prompts), batch_size):
        inputs = tokenizer.apply_chat_template(prompts[i:i + batch_size], add_generation_prompt=True, return_tensors="pt",
                                               return_dict=True, padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                                 pad_token_id=tokenizer.pad_token_id)
        gen = out[:, inputs["input_ids"].shape[1]:]
        outs += [tokenizer.decode(o, skip_special_tokens=True).strip() for o in gen]
        cut += [o[-1].item() not in (tokenizer.eos_token_id, tokenizer.pad_token_id) for o in gen]   # 長さの上限で打ち切られたか
    return outs, np.array(cut)

# 調整前 (= DPO の参照モデル) の対数尤度と、評価用の質問 100 件への応答
model.eval()
ref_c, ref_r = response_logps(model, test, "chosen"), response_logps(model, test, "rejected")
gen_prompts = [ex["prompt"] for ex in test[:100]]
before, cut_before = generate(model, gen_prompts)   # Colab の T4 で 2〜3 分
print("調整前に、選ばれた応答のほうが尤度が高い割合:", (ref_c > ref_r).mean())

調整前に、選ばれた応答のほうが尤度が高い割合: 0.48


In [5]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
args = DPOConfig(output_dir="dpo-out", beta=0.1, learning_rate=5e-5, num_train_epochs=1, per_device_train_batch_size=8,
                 warmup_steps=10, max_length=512, logging_steps=25, save_strategy="no", report_to="none",
                 gradient_checkpointing=False, bf16=BF16, fp16=not BF16, seed=0)   # DPOConfig は bf16=True が既定なので明示する
# ref_model を渡さずに peft_config を渡すと、LoRA を外した元のモデルが参照モデルになる (モデルを 2 つ読み込まずに済む)
trainer = DPOTrainer(model=model, args=args, train_dataset=Dataset.from_list(train), processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()
trainer.train()   # Colab の T4 で 10〜15 分程度
for log in trainer.state.log_history:
    if "rewards/accuracies" in log:
        print(f"step {log['step']:3d}  loss {log['loss']:.3f}  報酬の正解率 {log['rewards/accuracies']:.3f}  報酬の差 {log['rewards/margins']:.3f}")
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

trainable params: 10,076,160 || all params: 803,124,480 || trainable%: 1.2546


Step,Training Loss
25,0.701068
50,0.778530
75,0.711842
100,0.634453
125,0.716422
150,0.679537
175,0.588459
200,0.616448
225,0.616606
250,0.619532


step  25  loss 0.701  報酬の正解率 0.565  報酬の差 0.285
step  50  loss 0.779  報酬の正解率 0.610  報酬の差 0.414
step  75  loss 0.712  報酬の正解率 0.615  報酬の差 0.323
step 100  loss 0.634  報酬の正解率 0.660  報酬の差 0.507
step 125  loss 0.716  報酬の正解率 0.630  報酬の差 0.321
step 150  loss 0.680  報酬の正解率 0.615  報酬の差 0.445
step 175  loss 0.588  報酬の正解率 0.680  報酬の差 0.622
step 200  loss 0.616  報酬の正解率 0.640  報酬の差 0.536
step 225  loss 0.617  報酬の正解率 0.685  報酬の差 0.659
step 250  loss 0.620  報酬の正解率 0.665  報酬の差 0.589
GPU メモリの最大使用量: 9.2 GB


In [6]:
# 評価データでの選好の正解率。DPO の暗黙の報酬 r = β (log π(応答) − log π_ref(応答)) で、chosen と rejected を比べる
model = trainer.model.eval()
pol_c, pol_r = response_logps(model, test, "chosen"), response_logps(model, test, "rejected")
margin = args.beta * ((pol_c - ref_c) - (pol_r - ref_r))

def rate_ci(hits):   # 割合と、ブートストラップによる 95% 信頼区間
    boot = [hits[i].mean() for i in np.random.default_rng(0).integers(0, len(hits), (2000, len(hits)))]
    return f"{hits.mean():.3f} (95% 信頼区間 {np.percentile(boot, 2.5):.3f}〜{np.percentile(boot, 97.5):.3f}、{len(hits)} 件)"

print(f"評価データで、報酬が chosen > rejected になった割合: {rate_ci(margin > 0)}、報酬の差の平均 {margin.mean():.3f}")
print(f"選ばれた応答のほうが尤度が高い割合: 調整前 {(ref_c > ref_r).mean():.3f} → 調整後 {(pol_c > pol_r).mean():.3f}")
print(f"対数尤度の平均の変化: chosen {np.mean(pol_c - ref_c):+.2f}、rejected {np.mean(pol_r - ref_r):+.2f}")

評価データで、報酬が chosen > rejected になった割合: 0.680 (95% 信頼区間 0.615〜0.745、200 件)、報酬の差の平均 0.587
選ばれた応答のほうが尤度が高い割合: 調整前 0.480 → 調整後 0.485
対数尤度の平均の変化: chosen +12.82、rejected +6.95


In [7]:
# 同じ質問への応答を、調整の前後で比べる
after, cut_after = generate(model, gen_prompts)
for p, b, a in list(zip(gen_prompts, before, after))[:4]:
    print("質問:", p[0]["content"], "\n[前]", b[:250], "\n[後]", a[:250], "\n")
def n_tok(texts):
    return np.mean([len(tokenizer(t)["input_ids"]) for t in texts])

print(f"応答の平均の長さ: 調整前 {np.mean([len(s) for s in before]):.0f} 文字 ({n_tok(before):.0f} トークン) → "
      f"調整後 {np.mean([len(s) for s in after]):.0f} 文字 ({n_tok(after):.0f} トークン)")
print(f"上限の 384 トークンで打ち切られた割合: 調整前 {cut_before.mean():.2f} → 調整後 {cut_after.mean():.2f}")

質問: ローマのコンクリートはどのように作られたのですか？ 
[前] ローマのコンクリートは、その独特の耐久性と美しさで知られていますが、その作り方にはいくつかの重要なステップがあります。まず、ローマのコンクリートは主に石灰岩や砂、砂利などの自然素材から構成されています。これらの材料を混合し、水を加えて練り上げることで基礎となるブロックが造られます。

次に、コンクリートブロックは型枠を用いて作られます。この型枠は石やレンガでできており、特定の形やサイズに調整されます。ブロックは一つ一つ丁寧に組み立てられ、特に側面や底面には補強材が付けられます。これにより、強度が 
[後] ローマのコンクリートは、その独特の耐久性と美しさで知られています。ローマのコンクリートは主に「コンクリトゥド・ summa（ summa は「総体」を意味し、建物の全体を指す）」として知られており、その作り方にはいくつかの重要なステップがあります。

まず、コンクリートの基礎となる材料は砂、砂利、水、そして時には石灰岩や石膏が使われます。これらの材料をよく混ぜ合わせ、型枠に流し込んで固めます。このとき、型枠は木や石で作られ、非常に精密に設計されます。

次に、コンクリートが固まったら、内部の空気 

質問: AATIPとは何ですか？ 
[前] AAtip（Alternative Authentication Protocol for Internet Protection）は、インターネット保護のためのプロトコルの一つであり、特にファイアウォールやネットワークセキュリティにおいて使用されます。このプロトコルは、従来のIPsec（Internet Protocol Security）よりも軽量で高速な通信を可能にし、特にモバイルデバイスや低帯域幅環境での使用に適しています。AAtipは、認証、暗号化、およびアクセス制御を一つの統合された 
[後] AAtip（Alternative Authentication Protocol for Internet Transactions）とは、インターネットにおける認証プロトコルの一種で、特にウェブサービスやオンライン取引においてセキュリティを強化するために設計されています。AAtipは、従来のパスワードベースの認証方法に代替し、より安全で効率的な認証手

In [8]:
# 調整前と後の応答を、大きめのモデル (sarashina2.2-3b-instruct、MIT License) に比べさせる (LLM による判定)。
# 応答を A と B に並べ、次のトークンが「A」と「B」になる確率を比べる。位置の偏りを打ち消すため、順番を入れ替えて 2 回聞いて平均する
judge_name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
jtok = load_tokenizer(judge_name)
judge = AutoModelForCausalLM.from_pretrained(judge_name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()
id_a, id_b = [jtok(c, add_special_tokens=False)["input_ids"][-1] for c in ("A", "B")]   # 評価指標と統計的検定の章と同じ取り方
print("判定に使うトークン:", jtok.convert_ids_to_tokens([id_a, id_b]))

def p_first_better(q, x, y):
    def p_a(a, b):
        msg = [{"role": "user", "content": f"次の質問に対する 2 つの応答のうち、より役に立つ応答はどちらですか。A か B の 1 文字で答えてください。\n\n"
                                           f"質問: {q}\n\n応答 A: {a}\n\n応答 B: {b}"}]
        inputs = jtok.apply_chat_template(msg, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(judge.device)
        with torch.no_grad():
            logits = judge(**inputs).logits[0, -1].float()
        return torch.softmax(logits[[id_a, id_b]], -1)[0].item()
    return (p_a(x, y) + 1 - p_a(y, x)) / 2

# まず、判定役が人の選好 (評価データの chosen / rejected) とどれくらい一致するかを確かめる。Colab の T4 で 2〜3 分
agree = np.array([p_first_better(ex["prompt"][0]["content"], ex["chosen"][0]["content"], ex["rejected"][0]["content"]) > 0.5 for ex in test])
print("判定役と人の選好の一致率:", rate_ci(agree))

win = np.array([p_first_better(p[0]["content"], a, b) for p, b, a in zip(gen_prompts, before, after)])
print("調整後の応答が調整前より良いと判定された割合 (勝率):", rate_ci(win > 0.5))
longer = np.array([len(a) > len(b) for b, a in zip(before, after)])   # 判定役が長い応答を好んでいないか
print(f"勝率の内訳: 調整後のほうが長い {longer.sum()} 問で {(win[longer] > 0.5).mean():.3f}、"
      f"長くならなかった (短いか同じ) {(~longer).sum()} 問で {(win[~longer] > 0.5).mean():.3f}")
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

判定に使うトークン: ['A', 'B']
判定役と人の選好の一致率: 0.650 (95% 信頼区間 0.585〜0.715、200 件)
調整後の応答が調整前より良いと判定された割合 (勝率): 0.340 (95% 信頼区間 0.250〜0.430、100 件)
勝率の内訳: 調整後のほうが長い 29 問で 0.517、長くならなかった (短いか同じ) 71 問で 0.268
GPU メモリの最大使用量: 9.2 GB


- DPO (Direct Preference Optimization; Rafailov et al., 2023) は、選好データから報酬モデルを学習して強化学習 (PPO) で最適化する RLHF の手順を、1 つの分類のような損失に置き換えたもの。参照モデルからのずれを β で抑えながら、chosen の尤度を rejected より相対的に上げる。実際には、この章の「ベースモデルを LoRA で指示チューニングする (SFT)」のような指示チューニングの後に行う。
- 上の出力では、評価データ（学習に使っていない 200 組）で報酬が chosen > rejected になった割合は 0.680（95% 信頼区間 0.615〜0.745）で、偶然 (0.5) より高い。ただし DPO が上げるのは chosen と rejected の差だけで、対数尤度そのものは chosen も rejected も上がり（平均で +12.82 と +6.95）、「chosen のほうが尤度が高い」割合は 0.480 → 0.485 とほとんど変わらなかった。
- 一方、生成した応答を判定役のモデルに比べさせると、調整後が良いとされたのは 100 問中 0.340（95% 信頼区間 0.250〜0.430）で、調整前より悪いと判定された。応答は短くなり（平均 480 → 435 文字）、長くならなかった 71 問では勝率が 0.268 と特に低い。GPU の計算は実行ごとにわずかに変わることがあり、勝率は小数第 2 位で変わりうるので、100 問の信頼区間の幅で見る。選好の正解率が上がっても、生成が良くなるとは限らない。
- 原因は確かめていないが、学習データの応答は hh-rlhf の論文（2022 年）のモデルの出力を機械翻訳したもので、ここでは 400 文字以下に絞っており、このモデル自身の応答とは文体も長さも違う（方策外のデータ）。調整するモデル自身に応答を複数生成させ、それに人や強いモデルが選好を付けたデータ（方策上のデータ）を使う方法もある。
- 判定役 (3B) と人の選好の一致率は 0.650 程度で、判定はあくまで目安。A と B の順番を入れ替えて平均し、位置の偏りを打ち消している。判定役の信頼性の確かめ方は、LLM の評価と分析の章の「手元の LLM に生成文を採点させる (LLM-as-a-judge)」を参照。
- データ: hh-rlhf の元のデータセットカードには、選好（報酬）モデルの学習用で、対話エージェントを教師ありで学習するためのものではない（有害なモデルになりうる）と書かれている。DPO は報酬モデルを作らず、対話モデルそのものをこの応答で直接学習するので、この注意が当てはまる。ここで作るモデルは手法を試すためのもので、公開や実用には使わない。harmless-base には差別や暴力などの不快な内容が含まれるので、ここでは使っていない。日本語は DeepL による翻訳で、訳の崩れも混じる（上の 1 件目の「」の対応など）。
- `DPOConfig` は、`TrainingArguments` と違って `bf16=True`、学習率 1e-6、`gradient_checkpointing=True` が既定。T4 では bf16 が使えないので fp16 を明示し、LoRA では学習率を上げる（ここでは 5e-5）。β を大きくすると参照モデルから離れにくくなる。`loss_type="ipo"` などで損失を IPO などに変えられる。trl 1.18.0 で確かめた（版によって引数や既定値が変わることがある）。
- GPU メモリの最大使用量は学習時の 9.2GB で、3B の判定役を読み込んだ後も変わらなかった。sarashina2.2 のトークナイザーの修正 (`load_tokenizer`) については、この章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。